# Tutorial: create OSC metadata for subglacial lakes

This tutorial shows how to create a Open Science Catalog (OSC) metadata contribution with the EarthCODE Library.  Use the tutorial to structure your the script/notebook for your own contribution to the OSC.

You will see an example, how to:

1. Step 1. setup the local enviroment
2. Step 2. Add metadata
- a `Project` collection for the ESA research context;
- a `Product` collection for the discoverable dataset;
- a `Workflow` record for the code or method;
- an `Experiment` record linking the created `Workflow` and `Product`;
3. Step 3. Add STAC Item level medata for each of your files
- one file-level STAC Item for a Parquet asset;
4. Step 4. Run validation
- a final validation report for the local catalog checkout.
5. Step 5: Open a PR


This notebook uses made up IDs so it can be run without changing an existing published record. 

> Do not open a pull request from this example. 

## Step 1. Setup a local enviroment

You need an environment with the EarthCODE Library installed and a local fork or clone of `open-science-catalog-metadata`. The file-level metadata step also reads a small public Parquet file from object storage.


To see how to setup your enviroment in detail checkout the `guides/0.Prerequisites-local.ipynb` guide.

In summary, you should:
- Fork the open science catalog repository on github - [https://github.com/ESA-EarthCODE/open-science-catalog-metadata](https://github.com/ESA-EarthCODE/open-science-catalog-metadata)
- Clone the fork locally
- Install the earthcode-library



The save and validation helpers write into a local `open-science-catalog-metadata` instance.  Set `catalog_root` to that instance's path before you create any metadata records.

In [1]:
from datetime import datetime
from pathlib import Path

from earthcode.metadata_input_definitions import ProductCollectionMetadata
from earthcode.static import create_product_collection
from earthcode.git_add import save_product_collection_to_catalog

from earthcode.metadata_input_definitions import ProjectCollectionMetadata
from earthcode.static import create_project_collection
from earthcode.git_add import save_project_collection_to_osc
from earthcode.validator import validate_catalog

from earthcode.search import search
import geopandas as gpd
import pystac

In [2]:
catalog_root = Path("../../../open-science-catalog-metadata").resolve()

## Step 2. Create the catalog records

Start with the catalog-level records before adding file-level metadata.


### Create the Project metadata

A Project is the top-level OSC context for an ESA activity. In this step you provide the project ID, title, description, status, license, spatial and temporal coverage, science themes, Technical Officer contact, website links, and consortium members.

For your own contribution, replace the example values with the official project metadata. Keep IDs short, lowercase, and dash-separated.

See the `guides/1.Project.ipynb` guide for more information about the project collection.

In [3]:
from pathlib import Path

# A custom id of the project, it can be related to the title, i.e. - 4datlantic-ohc. Use dash "-" symbol to separare words in the id"
project_id = "4d-antarctica-v2"

# Specify the Title of your project. I.e. - 4DAtlantic-OHC. This should correspond to the title of the project as in the ESA contract.
project_title = "4D-ANTARCTICA-v2"

# A short description of the project:
project_description = "Ice sheets are a key component of the Earth system, impacting on global sea level, ocean circulation and bio geochemical processes. Significant quantities of liquid water are being produced and transported at the ice sheet surface, base, and beneath its floating sections, and this water is in turn interacting with the ice sheet itself. Despite the key role that hydrology plays on the ice sheet environment, there is still no global hydrological budget for Antarctica. There is currently a lack of global data on supra: and sub:glacial hydrology, and no systems are in place for continuous monitoring of it or its impact on ice dynamics. The overall aim of 4DAntarctica is to advance our understanding of the Antarctic Ice Sheets supra and sub:glacial hydrology, its evolution, and its role within the broader ice sheet and ocean systems. We designed our programme of work to address the following specific objectives: Creating and consolidating an unprecedented dataset composed of ice:sheet wide hydrology and lithospheric products, Earth Observation datasets, and state of the art ice sheet and hydrology models; Improving our understanding of the physical interaction between electromagnetic radiation, the ice sheet, and liquid water; Developing techniques and algorithms to detect surface and basal melting from satellite observations in conjunction with numerical modelling; Applying these new techniques at local sites and across the continental ice sheet to monitor water dynamics and derive new hydrology datasets; Performing a scientific assessment of Antarctic Ice Sheet hydrology and of its role in the current changes the continent is experiencing; Proposing a future roadmap for enhanced observation of Antarcticas hydrological cycle. To do so, the project will use a large range of Earth Observation missions (e.g. Sentinel 1, Sentinel 2, SMOS, CryoSat 2, GOCE, TanDEM X, AMSR2, Landsat, Icesat 2) coupled with ice:sheet and hydrological models. By the end of this project, the programme of work presented here will lead to a dramatically improved quantification of meltwater in Antarctica, an improved understanding of fluxes across the continent and to the ocean, and an improved understanding of the impact of the hydrological cycle on ice sheets mass balance, its basal environment, and its vulnerability to climate change."

# Project status: pick from - ongoing or completed
project_status = "completed"

# Overall license for all related data that will be uploaded from the project., i.e. CC-BY-4.0. See the note in the markdown cell above to consult full list of available licenses.
# If you have multiple licenses, you can pick 'various'
project_license = "proprietary"

# Define spatial extent of the project study area in epsg:4326
# if you have multiple disjoint study areas, specify the bounding box that covers all of them
# i.e project_w, project_s,  project_e, project_n = -180.0, -90.0, 180.0, 90.0
project_w = -180.0
project_s = -90.0
project_e = 180.0
project_n = 90.0

# The project start and end times
project_start_year = 2019
project_start_month = 9
project_start_day = 24

project_end_year = 2022
project_end_month = 10
project_end_day = 15

# Define the links to the project websites  
website_link = "http://www.4d-antarctica.org/" # link to project-dedicated website (hosted also on external servers)
eo4society_link = "https://eo4society.esa.int/projects/4dantarctica/" # link to the project website on eo4society projects list

# Define project themes, according to OSC ontology. Pick one or more from:
# - atmosphere, cryosphere, land, magnetosphere-ionosphere, oceans, solid-earth.
# See the list here: https://github.com/ESA-EarthCODE/open-science-catalog-metadata/blob/main/themes/catalog.json
project_themes = ["cryosphere"]

# provide the Name and e-mail address to ESA Technical Officer (TO) supporting your project:
to_name = "Diego Fernandez"
to_email = "diego.fernandez@esa.int"

# List the consortium members in a tuple with format (name, contact_email), for example - ('University A', "contact@universitya.fr")
consortium_members = [
    ("UNIVERSITY OF EDINBURGH (GB)", ""),
    ("CNR: RESEARCH INSTITUTE FOR GEO:HYDROLOGICAL PROTECTION IRPI (IT)", ""),
    ("DLR GERMAN AEROSPACE CENTER (DE)", ""),
    ("EARTHWAVE Ltd (GB)", ""),
    ("EIDGENOSSISCHE TECHNISCHE HOCHSCHULE ZURICH (ETH ZURICH) (CH)", ""),
    ("ENVEO ENVIRONMENTAL EARTH OBSERVATION GMBH (AT)", ""),
    ("SHEPHERD SPACE LTD. (GB)", ""),
    ("Technical University of Denmark (DK)", ""),
    ("UKRI Rutherford Appleton Laboratory (GB)", ""),
    ("UNIVERSITY OF LANCASTER ENVIROMENT CENTRE (GB)", ""),
    ("UNIVERSITY OF LEEDS", ""),
    ("SCHOOL OF EARTH AND ENVIRONMENT (GB)", ""),
    ("UNIVERSITE GRENOBLE ALPES (FR)", "")
]

Build the Project collection from the structured input. This turns the Python metadata object into the OSC/STAC collection that will be written to the catalog.


In [4]:
project_bbox = [[project_w, project_s, project_e, project_n]]

project_metadata = ProjectCollectionMetadata(
    project_id=project_id,
    project_title=project_title,
    project_description=project_description,
    project_status=project_status,
    project_license=project_license,
    project_bbox=project_bbox,
    project_start_datetime=datetime(project_start_year, project_start_month, project_start_day),
    project_end_datetime=datetime(project_end_year, project_end_month, project_end_day),
    project_themes=project_themes,
    to_name=to_name,
    to_email=to_email,
    consortium_members=consortium_members,
    website_link=website_link,
    eo4society_link=eo4society_link or None,
)

project_collection = create_project_collection(project_metadata)


In [5]:

# save the projection collection to the catalog fork
save_project_collection_to_osc(project_collection, catalog_root)


### Create the Product metadata

A Product describes the dataset that OSC users should be able to find and reuse. It links back to the Project and records the dataset title, description, license, keywords, spatial and temporal extent, region, themes, EO missions, variables, parameters, and DOI or access link.

A project can have multiple products.

Use the product extent and dates for the dataset itself, not just the wider project. If your product has several disconnected areas, provide one bounding box for each area.

See the `guides/2.Product.ipynb` guide for more information about the project collection.

In [6]:
# Define id, title, description, product status,license.
# A custom id of the product (must be different from project!), it can be related to the title, i.e. - 4datlantic-ohc-dataset. Use dash "-" symbol to separare words in the id"
product_id = "subglacial-lakes-boundries-2"
product_title = "Subglacial lakes boundaries and elevation time series"
product_description = "85 active Antarctic subglacial lakes detected from a decade of CryoSat-2 radar altimetry. Documenting draining and filling events and time-varying boundaries of subglacial lake activity. Ice surface elevation change (dz) for the period October 2010 to July 2020."
# Product status: pick from - ongoing or completed
product_status = "completed"

# Define the product license. i.e. CC-BY-4.0. See the note in the markdown cell above to consult full list of available licenses. 
# If you have multiple licenses, you can pick 'various'
product_license = 'CC-BY-4.0'

# Define at most five keywords for the product. You can use any short text, that allow users to discover your product.
product_keywords = [ 
    "Subglacial lakes",
    "Glaciers",
    "Glacial Lake Extent",
    "Antarctica"
] 

# Define spatial extent of PRODUCT/DATASET in epsg:4326. If the dataset covers discontinuous regions, \
# add the bounding box boundaries for each
# i..e 
# - a dataset with global coverage is:product_s product_w, product_n, product_e = [-180.0], [-90.0], [180.0], [90.0]
# - a dataset with multiple disjoint regions is: 
#       product_s product_w, product_n, product_e = [-180.0, -180.0], [-90.0, -90.0], [180.0, 180.0], [90.0, 90.0]
# Use bounds from your existing GeoDataFrame, ensuring EPSG:4326 for OSC product extent


# Keep variable names expected by the notebook's next cell (zip order is used as bbox order)
product_w = [-180.0]
product_s = [-90.0]
product_e = [180.0]
product_n = [-60.0]



# Define the temporal extent of PRODUCT/ DATASET
start_dt = "2010-07-01T00:00:00Z"
end_dt = "2020-07-01T00:00:00Z"

product_start_year, product_start_month, product_start_day = (
    2010, 7, 1
)
product_end_year, product_end_month, product_end_day = (
    2020, 7, 1
)


# Define the semantic region covered by this product, i.e. Belgium, Global etc. 
product_region = "Antarctica"

# Define product themes i.e. land. Pick one or more from:
# - atmosphere, cryosphere, land, magnetosphere-ionosphere, oceans, solid-earth.
# See the list here: https://github.com/ESA-EarthCODE/open-science-catalog-metadata/blob/main/themes/catalog.json
product_themes = ["cryosphere"]

# Define the eo-misison(s) used to generate the product. i.e. - "sentinel-2"
# Pick one or more from - https://github.com/ESA-EarthCODE/open-science-catalog-metadata/tree/main/eo-missions
# i.e. product_missions = ['in-situ-observations', 'grace', 'numerical-models']
product_missions = ["cryosat-2"]

# Define variables describing at best your Product/ dataset: 
# Pick one or more from from https://github.com/ESA-EarthCODE/open-science-catalog-metadata/tree/main/variables
# Please see the comment in the markdown cell above to select a right variable and provide it in the correct format i.e. i.e. "crop-yield-forecast"
product_variables = ["subglacial-lake"]
# Define the parameters describing your product in standardised CF convention format: i.e. "leaf_area_index". 
# Please see the description in the markdown cell above to select a right parameter and provide it in the correct format
product_parameters = ["subglacial-lake-extent"]

# Provide DOI number assigned to your product, i.e. "https://doi.org/10.57780/s3d-83ad619". If your product does not have one, type: 'None'
# See description in the markdown cell to request the DOI if needed. 
product_doi = "https://doi.org/10.5281/zenodo.16330564"

# Define the related project id and title
# These must match the new or an already existing project in the catalog! Alteratively correct links cannot be produced!  
# See the previous 'project metatdata' step, and copy and paste these parameters according to your previous inputs. 
project_id = project_id
project_title = project_title




Build the Product collection from the input fields. The product must reference the Project created above so OSC can connect the dataset to its research context.


In [7]:
# create a product
product_bbox = [[w, s, e, n] for s, w, n, e in zip(product_s, product_w, product_n, product_e)]

product_metadata = ProductCollectionMetadata(
    product_id=product_id,
    product_title=product_title,
    product_description=product_description,
    product_bbox=product_bbox,
    product_start_datetime=datetime(product_start_year, product_start_month, product_start_day),
    product_end_datetime=datetime(product_end_year, product_end_month, product_end_day),
    product_license=product_license,
    product_keywords=product_keywords,
    product_status=product_status,
    product_region=product_region,
    product_themes=product_themes,
    product_missions=product_missions,
    product_variables=product_variables,
    project_id=project_id,
    project_title=project_title,
    product_parameters=product_parameters,
    product_doi=product_doi,
    license_link=None,
    access_link=product_doi
)

product_collection = create_product_collection(product_metadata)

In [8]:
save_product_collection_to_catalog(product_collection, catalog_root)

### Create the Workflow metadata

A Workflow records the code, method, or processing logic associated with the product. This helps reviewers and future users understand how the dataset can be reproduced or inspected.

See the `guides/3.Workflow.ipynb` guide for more information about the project collection.

The `codeurl` should point to a stable code repository, archive, or DOI; the formats and themes should match what the workflow actually consumes and produces.

In [9]:
from datetime import datetime
from pathlib import Path

from earthcode.metadata_input_definitions import WorkflowMetadata
from earthcode.static import create_workflow_record
from earthcode.git_add import save_workflow_record_to_osc
from earthcode.validator import validate_catalog

In [10]:
# BASIC INFORMATION ABOUT THE WORKFLOW
# Unique ID for the visualization/reproduction script
workflow_id = "sgl-time-series-reproduction-2"
workflow_title = "Subglacial Lake Time Series Visualization Workflow"
workflow_description = "Workflow for reproducing ice surface elevation change figures from time series data." 

# Define at most five keywords for the workflow
workflow_keywords = [
    "Subglacial lakes",
    "Glaciers",
    "Glacial Lake Extent",
    "Antarctica"
] 

# Define the license of the workflow
workflow_license = 'MIT'

# Input/Output formats: The script consumes CSV/Parquet and outputs figures (PNG/PDF)
workflow_formats = ['csv', 'geoparquet', 'png']


# Theme
workflow_themes = ['cryosphere']

# Contact info (Using corresponding author from Wilson et al., 2025)
workflow_contracts_info = [('Sally F. Wilson', "n/a")]

# URL to the code repository/DOI
codeurl = 'https://doi.org/10.5281/zenodo.16330564'


# Optional workflow record fields
workflow_doi = None
workflow_w = -180.0
workflow_s = -90.0
workflow_e = 180.0
workflow_n = 90.0
workflow_start_year = 2021
workflow_start_month = 1
workflow_start_day = 1
workflow_end_year = 2021
workflow_end_month = 12
workflow_end_day = 31
include_workflow_bbox = False
include_workflow_time = False


In [11]:
workflow_metadata = WorkflowMetadata(
    workflow_id=workflow_id,
    workflow_title=workflow_title,
    workflow_description=workflow_description,
    workflow_license=workflow_license,
    workflow_keywords=workflow_keywords,
    workflow_formats=workflow_formats,
    workflow_themes=workflow_themes,
    codeurl=codeurl,
    project_id=project_id,
    project_title=project_title,
    workflow_doi=workflow_doi,
    workflow_bbox=[[workflow_w, workflow_s, workflow_e, workflow_n]] if include_workflow_bbox else None,
    workflow_start_datetime=datetime(workflow_start_year, workflow_start_month, workflow_start_day) if include_workflow_time else None,
    workflow_end_datetime=datetime(workflow_end_year, workflow_end_month, workflow_end_day) if include_workflow_time else None,
)

workflow_record = create_workflow_record(workflow_metadata)


In [12]:
save_workflow_record_to_osc(workflow_record, catalog_root)


### Add an experiment

An experiment links a product and a workflow together, showing that a particular dataset is the output of a specific run of an algorithm.

More information is available in `docs/guides/5.Experiment.ipynb`.

In [13]:
from datetime import datetime
from pathlib import Path

from earthcode.metadata_input_definitions import ExperimentMetadata
from earthcode.static import create_experiment_record
from earthcode.git_add import save_experiment_record_to_osc

In [14]:
# experiment info
# Experiment id
experiment_id = workflow_id = "sgl-time-series-reproduction-experiment"
experiment_title = "Subglacial Lake Time Series Visualization Experiment"
experiment_description = "Experiment for the subglacial dataset." 

experiment_license = "CC-BY-4.0"
experiment_keywords = [
    "Subglacial lakes",
    "Glaciers",
    "Glacial Lake Extent",
    "Antarctica"
]

# Define the input/output formats that this experiment works with
# i.e. GeoTIFF, Zarr, netCDF, etc
experiment_formats = ['safefile', "parquet"]

# Define themes i.e. land. Pick one or more from:
# - atmosphere, cryosphere, land, magnetosphere-ionosphere, oceans, solid-earth.
experiment_themes = ["cryosphere"]

# link to the specification of the input parameters for the experiment
experiment_input_parameters_link = codeurl
# link to the enviroment in which the experiment was performed
experiment_enviroment_link = codeurl

## ID and title of the associated workflow
workflow_id = workflow_id
workflow_title = workflow_title

## ID and title title of the associated product
product_id = product_id
product_title = product_title

# Optional experiment record fields
experiment_contacts = None

experiment_w = -180.0
experiment_s = -90.0
experiment_e = 180.0
experiment_n = 90.0
experiment_start_year = 2021
experiment_start_month = 1
experiment_start_day = 1
experiment_end_year = 2021
experiment_end_month = 12
experiment_end_day = 31

include_experiment_bbox = False
include_experiment_time = False

In [15]:
catalog_root = Path(catalog_root)

experiment_metadata = ExperimentMetadata(
    experiment_id=experiment_id,
    experiment_title=experiment_title,
    experiment_description=experiment_description,
    experiment_license=experiment_license,
    experiment_keywords=experiment_keywords,
    experiment_formats=experiment_formats,
    experiment_themes=experiment_themes,
    experiment_input_parameters_link=experiment_input_parameters_link,
    experiment_enviroment_link=experiment_enviroment_link,
    workflow_id=workflow_id,
    workflow_title=workflow_title,
    product_id=product_id,
    product_title=product_title,
    contacts=experiment_contacts,
    experiment_bbox=[[experiment_w, experiment_s, experiment_e, experiment_n]] if include_experiment_bbox else None,
    experiment_start_datetime=datetime(experiment_start_year, experiment_start_month, experiment_start_day) if include_experiment_time else None,
    experiment_end_datetime=datetime(experiment_end_year, experiment_end_month, experiment_end_day) if include_experiment_time else None,
)

experiment_record = create_experiment_record(experiment_metadata)


In [16]:
save_experiment_record_to_osc(experiment_record, catalog_root)

## Step 3. Add file-level metadata as STAC Items

Product metadata describes the dataset as a whole. STAC Item metadata describes the individual files or assets that make up that dataset.

In this section you will create one STAC Item for a small Parquet asset, add projection and column metadata, attach the asset URL, validate the item, convert it to the EarthCODE-flavoured item, and save it under the product collection. 

If your own contribution has multiple files, you will have to create a STAC item for each file and add it to a collection.

See the guides for specific examples of how to generate metadata for the three most common data types:
- Parquet - `docs/guides/parquet_file_metadata.ipynb`
- Zarr - `docs/guides/zarr_file_metadata.ipynb`
- COG - `docs/guides/cog_file_metadata.ipynb`

In [17]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box, mapping
import pystac
from pystac.extensions.projection import ProjectionExtension
import datetime
import fsspec
import io

In [18]:
data_url = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/polar_cube_datasets/supraglacial_lakes/WAIS_Jan_2017_Polygons.parquet"


In [19]:
# this is a small file, so we read it in one go
with fsspec.open(data_url, mode="rb") as f:
    # Read the entire streaming file into memory
    file_bytes = f.read()
    
    # Wrap it in a BytesIO object, which PyArrow can seek through
    buffer = io.BytesIO(file_bytes)
    
    # Read the parquet from the memory buffer
    gdf = gpd.read_parquet(buffer)

gdf

,POLY_AREA,CENTROID_X,CENTROID_Y,Area,Shape_Leng,Shape_Area,Bedrock,Coast,GL_ON_IS,Location,...,PolsbyPopp,Fractal,Reock,Schwartzbe,LW_Ratio,Feature_Cl,Elevation,Slope,Speed,geometry
0,2250.000000,-1.122141e+06,-1.179198e+06,2298.467416,240.00000,2250.000000,439.236720,16041.380634,0.000000,Grounded Ice,...,0.490874,1.408355,0.506300,0.700624,0.857143,Lake,360,1.884210,61.69920,"POLYGON ((-1122127.5 -1179172.5, -1122127.5 -1..."
1,25200.000000,-1.122477e+06,-1.178988e+06,25742.726268,2160.00000,25200.000000,415.759238,16085.345499,0.000000,Grounded Ice,...,0.067874,1.319976,0.052796,0.260526,0.114516,Channel,360,1.884210,60.73470,"MULTIPOLYGON (((-1122607.5 -1178917.5, -112265..."
2,675.000000,-1.122930e+06,-1.178760e+06,689.534115,120.00000,675.000000,740.341624,16507.511953,0.000000,Grounded Ice,...,0.589049,1.360778,0.381972,0.767495,0.333333,Lake,388,2.731610,39.97260,"POLYGON ((-1122907.5 -1178767.5, -1122952.5 -1..."
3,11250.000000,-1.123134e+06,-1.178679e+06,11492.233178,930.00000,11250.000000,770.661893,16537.853558,0.000000,Grounded Ice,...,0.163454,1.364722,0.112676,0.404295,0.186667,Lake,388,2.731610,46.55940,"POLYGON ((-1123267.5 -1178602.5, -1123267.5 -1..."
4,198.750411,-1.809312e+06,7.091160e+05,200.159238,59.81225,198.750411,14427.999008,134873.872142,418.393202,Ice Shelf,...,0.698132,1.293517,0.509294,0.835543,0.500000,Lake,37,0.121964,9.98336,"POLYGON ((-1809304.184 709123.549, -1809311.33..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10473,2025.000000,2.573867e+05,-7.738833e+05,2121.704652,240.00000,2025.000000,0.000000,496657.687060,0.000000,Grounded Ice,...,0.441786,1.389131,0.449199,0.664670,0.666667,Lake,502,7.215030,47.97920,"POLYGON ((257407.5 -773857.5, 257407.5 -773872..."
10474,450.000000,2.574525e+05,-7.738425e+05,471.491754,120.00000,450.000000,0.000000,496703.680279,0.000000,Grounded Ice,...,0.392699,1.276085,0.318310,0.626657,0.500000,Lake,502,7.215030,47.97920,"MULTIPOLYGON (((257452.5 -773842.5, 257452.5 -..."
10475,3375.000000,2.576260e+05,-7.736940e+05,3536.188110,480.00000,3375.000000,74.210787,496768.404299,0.000000,Grounded Ice,...,0.184078,1.315911,0.169014,0.429043,0.333333,Lake,471,7.997810,56.07010,"MULTIPOLYGON (((257587.5 -773752.5, 257587.5 -..."
10476,6750.000000,2.577890e+05,-7.736955e+05,7072.372180,630.00000,6750.000000,135.624326,496743.461225,0.000000,Grounded Ice,...,0.213714,1.367931,0.232910,0.462292,0.444444,Lake,471,7.997810,56.07010,"POLYGON ((257827.5 -773677.5, 257827.5 -773707..."


In [20]:
# read the dataset spatial and temporal boundary
minx, miny, maxx, maxy = gdf.total_bounds
start, end = (datetime.datetime(2017, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
 datetime.datetime(2017, 12, 31, 0, 0, tzinfo=datetime.timezone.utc))

item = pystac.Item(
    id="supraglacial-lakes-boundries-only",
    geometry=mapping(box(minx, miny, maxx, maxy)),
    bbox=[minx, miny, maxx, maxy],
    datetime=start,
    properties={'title': 'Supraglacial lakes boundaries'},
    
)

proj = ProjectionExtension.ext(item, add_if_missing=True)
proj.code = gdf.crs.to_string()          # e.g. "EPSG:3031"
proj.wkt2 = gdf.crs.to_wkt()
proj.bbox = [minx, miny, maxx, maxy]    # native CRS bbox
proj.geometry = mapping(box(minx, miny, maxx, maxy))

item.extra_fields['title'] = 'Subglacial lakes boundaries data'
item

<Item id=supraglacial-lakes-boundries-only>

In [21]:
asset = pystac.Asset(href=data_url, media_type="application/x-parquet", roles=["data"])

asset.extra_fields["parquet:columns"] = [
    {
        "name": "POLY_AREA",
        "type": "float",
        "long_name": "Polygon area",
        "description": (
            "Area of the supraglacial lake polygon as provided in the source vector dataset. "
            "This may duplicate or closely correspond to Shape_Area or Area, depending on the "
            "source processing workflow."
        ),
        "unit": "m2",
    },
    {
        "name": "CENTROID_X",
        "type": "float",
        "long_name": "Lake centroid longitude",
        "description": (
            "X coordinate of the supraglacial lake polygon centroid. For Antarctic geographic "
            "datasets this is typically longitude in decimal degrees, unless the source layer "
            "uses a projected coordinate reference system."
        ),
        "unit": "degree",
    },
    {
        "name": "CENTROID_Y",
        "type": "float",
        "long_name": "Lake centroid latitude",
        "description": (
            "Y coordinate of the supraglacial lake polygon centroid. For Antarctic geographic "
            "datasets this is typically latitude in decimal degrees, unless the source layer "
            "uses a projected coordinate reference system."
        ),
        "unit": "degree",
    },
    {
        "name": "Area",
        "type": "float",
        "long_name": "Supraglacial lake area",
        "description": (
            "Surface area of the mapped supraglacial lake feature. Used to describe lake size "
            "and to support derived metrics such as volume estimates and area-perimeter ratios."
        ),
        "unit": "m2",
    },
    {
        "name": "Shape_Leng",
        "type": "float",
        "long_name": "Lake polygon perimeter",
        "description": (
            "Perimeter or boundary length of the supraglacial lake polygon geometry."
        ),
        "unit": "m",
    },
    {
        "name": "Shape_Area",
        "type": "float",
        "long_name": "Lake polygon shape area",
        "description": (
            "Area of the lake polygon geometry calculated by the source GIS software. This may "
            "duplicate or closely correspond to POLY_AREA or Area."
        ),
        "unit": "m2",
    },
    {
        "name": "Bedrock",
        "type": "float",
        "long_name": "Distance to exposed bedrock",
        "description": (
            "Distance from the supraglacial lake feature to the nearest mapped exposed bedrock "
            "or rock outcrop, where available."
        ),
        "unit": "m",
    },
    {
        "name": "Coast",
        "type": "float",
        "long_name": "Distance to coast",
        "description": (
            "Distance from the supraglacial lake feature to the Antarctic coastline or ice-shelf "
            "front used in the source analysis."
        ),
        "unit": "m",
    },
    {
        "name": "GL_ON_IS",
        "type": "boolean",
        "long_name": "Grounding-line-on-ice-shelf flag",
        "description": (
            "Flag indicating whether the supraglacial lake is located on an ice shelf or in a "
            "grounding-line-related ice-shelf setting, according to the source classification."
        ),
        "unit": None,
    },
    {
        "name": "Location",
        "type": "string",
        "long_name": "Lake location class",
        "description": (
            "Textual location category or regional setting assigned to the supraglacial lake, "
            "such as ice sheet, ice shelf, grounding-zone, coastal, or named regional grouping, "
            "depending on the source classification."
        ),
        "unit": None,
    },
    {
        "name": "GL",
        "type": "float",
        "long_name": "Distance to grounding line",
        "description": (
            "Distance from the supraglacial lake feature to the Antarctic grounding line used "
            "in the source analysis."
        ),
        "unit": "m",
    },
    {
        "name": "Volume",
        "type": "float",
        "long_name": "Estimated lake volume",
        "description": (
            "Estimated water volume of the supraglacial lake feature, typically derived from "
            "mapped lake area and an empirical or remotely sensed depth relationship."
        ),
        "unit": "m3",
    },
    {
        "name": "A_P_Ratio",
        "type": "float",
        "long_name": "Area-perimeter ratio",
        "description": (
            "Ratio of lake polygon area to perimeter, used as a compactness or shape metric for "
            "the mapped supraglacial lake."
        ),
        "unit": "m",
    },
    {
        "name": "REMA_Max",
        "type": "float",
        "long_name": "Maximum REMA elevation",
        "description": (
            "Maximum surface elevation from the Reference Elevation Model of Antarctica within "
            "or associated with the supraglacial lake polygon."
        ),
        "unit": "m",
    },
    {
        "name": "REMA_Min",
        "type": "float",
        "long_name": "Minimum REMA elevation",
        "description": (
            "Minimum surface elevation from the Reference Elevation Model of Antarctica within "
            "or associated with the supraglacial lake polygon."
        ),
        "unit": "m",
    },
    {
        "name": "PolsbyPopp",
        "type": "float",
        "long_name": "Polsby-Popper compactness",
        "description": (
            "Shape compactness metric for the lake polygon based on the ratio of polygon area "
            "to the area of a circle with the same perimeter. Values closer to 1 indicate a more "
            "compact, circular shape."
        ),
        "unit": "1",
    },
    {
        "name": "Fractal",
        "type": "float",
        "long_name": "Fractal dimension shape index",
        "description": (
            "Shape-complexity metric describing the irregularity of the supraglacial lake polygon "
            "boundary. Higher values generally indicate a more complex or convoluted outline."
        ),
        "unit": "1",
    },
    {
        "name": "Reock",
        "type": "float",
        "long_name": "Reock compactness",
        "description": (
            "Shape compactness metric comparing the polygon area with the area of its minimum "
            "bounding circle. Values closer to 1 indicate a more compact shape."
        ),
        "unit": "1",
    },
    {
        "name": "Schwartzbe",
        "type": "float",
        "long_name": "Schwartzberg compactness",
        "description": (
            "Shape compactness metric comparing the polygon perimeter with the circumference of "
            "a circle of equal area. Values closer to 1 generally indicate a more compact, "
            "circular polygon."
        ),
        "unit": "1",
    },
    {
        "name": "LW_Ratio",
        "type": "float",
        "long_name": "Length-width ratio",
        "description": (
            "Ratio of lake length to lake width, used to describe elongation of the supraglacial "
            "lake polygon."
        ),
        "unit": "1",
    },
    {
        "name": "Feature_Cl",
        "type": "string",
        "long_name": "Feature class",
        "description": (
            "Classification assigned to the mapped feature in the source dataset. For this asset, "
            "features represent supraglacial lakes, but this field may preserve source-level "
            "feature class labels or quality classes."
        ),
        "unit": None,
    },
    {
        "name": "Elevation",
        "type": "float",
        "long_name": "Lake surface elevation",
        "description": (
            "Representative surface elevation associated with the supraglacial lake feature, "
            "likely derived from REMA or another Antarctic digital elevation model."
        ),
        "unit": "m",
    },
    {
        "name": "Slope",
        "type": "float",
        "long_name": "Surface slope",
        "description": (
            "Representative ice-surface slope associated with the supraglacial lake feature."
        ),
        "unit": "degree",
    },
    {
        "name": "Speed",
        "type": "float",
        "long_name": "Ice surface velocity",
        "description": (
            "Representative ice-flow speed at or near the supraglacial lake feature, derived from "
            "an Antarctic ice-velocity product used in the source analysis."
        ),
        "unit": "m yr-1",
    },
    {
        "name": "geometry",
        "type": "geometry",
        "long_name": "Supraglacial lake polygon geometry",
        "description": (
            "Polygon geometry representing the mapped supraglacial lake surface extent. Stored as "
            "the GeoParquet geometry column."
        ),
        "unit": None,
    },
]

item.add_asset('data', asset)
item.validate()
item

<Item id=supraglacial-lakes-boundries-only>

### Save the item under the Product collection

The generic STAC Item now needs the EarthCODE catalog links that connect it to the product. Convert it, then save it into the product collection in your local catalog checkout.


In [22]:

from pathlib import Path
from earthcode.static import generic_stac_item_to_earthcode_stac_item
from earthcode.git_add import save_item_to_product_collection

In [23]:
# convert to earthcode item with a reference to a specific open science catalog collection
earthcode_item  = generic_stac_item_to_earthcode_stac_item(item, product_id, asset_key='data')

In [24]:
# save the item to the collection
save_item_to_product_collection(earthcode_item, product_id, catalog_root)

## Step 4. Validate the local catalog changes

Run validation after all project, product, workflow, experiment, and file-level metadata for your real contribution have been saved to the open-science-catalog-metadata directory.

The tutorial raises an error if validation finds problems, so you do not continue with a broken local catalog.


In [ ]:
from earthcode.validator import validate_catalog

errors, error_files = validate_catalog(catalog_root)
if errors or error_files:
    raise AssertionError(f"Catalog validation failed. errors={len(errors)} files={len(error_files)}")

If validation fails, inspect the returned `errors` and `error_files` to find the records that need to be fixed.


In [ ]:
errors

[[<ValidationError: "'cc-by-4.0' is not one of ['various', 'various open source', 'proprietary', '0BSD', 'AAL', 'Abstyles', 'AdaCore-doc', 'Adobe-2006', 'Adobe-Glyph', 'Adobe-Utopia', 'ADSL', 'AFL-1.1', 'AFL-1.2', 'AFL-2.0', 'AFL-2.1', 'AFL-3.0', 'Afmparse', 'AGPL-1.0-only', 'AGPL-1.0-or-later', 'AGPL-3.0-only', 'AGPL-3.0-or-later', 'Aladdin', 'AMDPLPA', 'AML', 'AMPAS', 'ANTLR-PD', 'ANTLR-PD-fallback', 'Apache-1.0', 'Apache-1.1', 'Apache-2.0', 'APAFML', 'APL-1.0', 'App-s2p', 'APSL-1.0', 'APSL-1.1', 'APSL-1.2', 'APSL-2.0', 'Arphic-1999', 'Artistic-1.0', 'Artistic-1.0-cl8', 'Artistic-1.0-Perl', 'Artistic-2.0', 'ASWF-Digital-Assets-1.0', 'ASWF-Digital-Assets-1.1', 'Baekmuk', 'Bahyph', 'Barr', 'Beerware', 'Bitstream-Charter', 'Bitstream-Vera', 'BitTorrent-1.0', 'BitTorrent-1.1', 'blessing', 'BlueOak-1.0.0', 'Boehm-GC', 'Borceux', 'Brian-Gladman-3-Clause', 'BSD-1-Clause', 'BSD-2-Clause', 'BSD-2-Clause-Patent', 'BSD-2-Clause-Views', 'BSD-3-Clause', 'BSD-3-Clause-Attribution', 'BSD-3-Clause-C

In [ ]:
error_files

[PosixPath('/home/krasen/open-science-catalog-metadata/experiments/sgl-time-series-reproduction-experiment/record.json')]

## Step 5. Use the pattern for a real pull request

To finish your contribution, you have to:
- Commit the changed files to your local  `open-science-catalog-metadata` clone.
- Open a pull request against the main  `open-science-catalog-metadata` repository
- Then your PR will be reviewed and after any requested changes are made, merged into the Open Science Catalog repository.


 > ** Do not open a pull request with this tutorial output.  **


More details on openning the PR is available in the `guides/0.Prerequisites-local.ipynb` guide


## Next steps:
For a real OSC contribution, repeat the same sequence with your own metadata and stable data URLs:

1. Create Project metadata.
2. Create the Product metadata for each dataset and link it all Products to your Project.
3. Add Workflow metadata for reusable code or processing logic.
4. Add Experiment metadata when a specific workflow run generated the product.
5. Add file-level STAC Items for the all of your data files.
6. Run validation and fix any reported errors.
7. Open a pull request against `open-science-catalog-metadata` with only the intended metadata changes.

The pull request should explain what is being added, where the data is hosted, which licenses apply, and any review notes the EarthCODE Data Steward team needs.

